In [ ]:
# This example demonstrates concept of Prompt Chaining. Prompt Chaining is a technique where the output of one prompt is used as the input for another prompt, allowing for more complex and multi-step interactions with language models.
# In this example, we will create a simple workflow that takes topic as input, generates a detailed summary or the outline of the topic and generate a blog post based on the summary or outline. 
# The workflow will consist of two nodes: one for generating the summary and another for generating the blog post.

In [34]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import re

In [3]:
# Load environment variables from .env file
load_dotenv()

True

In [4]:
# Create LLM instance
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)

In [16]:
# Define a state of the workflow/Graph. This state will be passed between nodes in the workflow.
class BlogState(TypedDict):
    topic: str
    summary: str
    blog_post: str
    score: int

In [6]:
# Write a function to generate a summary based on the topic. This function will be used as a node in the workflow.
def generate_summary(state: BlogState) -> BlogState:
    topic = state['topic']
    prompt = f"Generate a detailed summary or outline for the following topic: {topic}"
    response = llm.invoke(prompt)
    state['summary'] = response.content
    return state

In [7]:
# Write a function to generate a blog post based on the summary. This function will be used as a node in the workflow.
def generate_blog_post(state: BlogState) -> BlogState:
    topic = state['topic']
    summary = state['summary']
    prompt = f"Generate a blog post on the topic - {topic} and based on the following summary: {summary}"
    response = llm.invoke(prompt)
    state['blog_post'] = response.content
    return state

In [35]:
# Evaluate the blog post and assign a score. This function will be used as a node in the workflow.
def generate_score(state: BlogState) -> BlogState:
    summary = state['summary']
    blog_post = state['blog_post']
    prompt = (
        "Evaluate the blog post based on the summary and assign a score between 1 and 10, "
        "where 10 is best and 1 is worst. Return only the number. "
        f"Summary: {summary} Blog Post: {blog_post}"
    )
    response = llm.invoke(prompt)
    response_text = response.content.strip()
    match = re.search(r"\b(10|[1-9])\b", response_text)
    if not match:
        raise ValueError(f"Could not parse score from model output: {response_text}")
    state['score'] = int(match.group(1))
    return state

In [36]:
# Create the State Graph
graph = StateGraph(BlogState)

In [37]:
# Add nodes to the graph (safe to rerun)
if 'Generate_Summary' not in graph.nodes:
    graph.add_node('Generate_Summary', generate_summary)
if 'Generate_Blog_Post' not in graph.nodes:
    graph.add_node('Generate_Blog_Post', generate_blog_post)

In [38]:
if 'Generate_Score' not in graph.nodes:
    graph.add_node('Generate_Score', generate_score)

In [39]:
# Add Edges to the graph to define the flow of the workflow
graph.add_edge(START, 'Generate_Summary')
graph.add_edge('Generate_Summary', 'Generate_Blog_Post')
graph.add_edge('Generate_Blog_Post', 'Generate_Score')
# The workflow ends after generating the score
graph.add_edge('Generate_Score', END)

In [40]:
# Compile the graph to create a workflow
compile_graph = graph.compile()

In [41]:
# Execute the workflow with a sample topic
input_state = {'topic': 'Future of AI'}
final_state = compile_graph.invoke(input_state)

In [42]:
print("Summary of the topic:\n", final_state['summary'])

Summary of the topic:
 I. Introduction
    A. Definition of AI
    B. Importance of AI in modern society
II. Current state of AI technology
    A. Examples of AI applications in various industries
    B. Challenges and limitations of current AI technology
III. Potential future advancements in AI
    A. Development of more advanced machine learning algorithms
    B. Integration of AI with other emerging technologies (e.g. IoT, blockchain)
    C. Expansion of AI applications into new industries
IV. Ethical and societal implications of advanced AI
    A. Impact on the job market and workforce
    B. Privacy concerns related to AI data collection and analysis
    C. Potential for bias and discrimination in AI algorithms
V. Strategies for ensuring responsible development and deployment of AI
    A. Government regulations and oversight
    B. Ethical guidelines for AI developers and companies
    C. Promoting diversity and inclusivity in AI research and development
VI. Conclusion
    A. Reca

In [43]:
print("Blog post on the topic:\n", final_state['blog_post'])

Blog post on the topic:
 As technology continues to advance at a rapid pace, the future of artificial intelligence (AI) holds great promise and potential. AI, defined as the simulation of human intelligence processes by machines, has already made a significant impact on modern society, revolutionizing industries such as healthcare, finance, and transportation. From autonomous vehicles to personalized medical treatments, AI has the ability to streamline processes, improve efficiency, and enhance decision-making.

Despite the many benefits of AI, the current state of AI technology still faces challenges and limitations. While AI has made significant advancements in recent years, there are still issues related to bias in algorithms, data privacy concerns, and the potential impact on the job market. However, the future of AI holds exciting possibilities for further advancements and innovations.

One potential area of growth in AI technology is the development of more advanced machine learn

In [44]:
print("Score of the blog post:\n", final_state['score'])

Score of the blog post:
 8
